# 3장 데이터 살펴보고 준비하기 — 탐색, 빈 값, 범주형, 스케일링, 데이터 누수, 파이프라인
『머신러닝과 딥러닝 기초』 실습 노트북

남극 펭귄 데이터로 다음을 해 봅니다.
1. 요약 통계와 그래프로 데이터 살펴보기
2. 상관관계와 심슨의 역설
3. 빈 값, 범주형 데이터 처리
4. 스케일링으로 펭귄 성별 맞히기 정확도 높이기
5. 데이터 누수를 막는 파이프라인

위에서부터 차례로 실행하세요.

## 1. 판다스로 데이터 훑어보기
`describe()`는 숫자 열마다 개수, 평균, 표준편차, 최솟값, 사분위수, 최댓값을 계산합니다.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("penguins")
df.describe().round(1)

범주형 열은 범주마다 몇 개인지 셉니다.

In [ ]:
df["island"].value_counts()

`groupby()`로 종별 평균을 구합니다.

In [ ]:
cols = ["flipper_length_mm", "body_mass_g"]
df.groupby("species")[cols].mean().round(1)

## 2. 그래프로 살펴보기
### 히스토그램

In [ ]:
sns.histplot(data=df, x="flipper_length_mm", hue="species", binwidth=3)
plt.show()

### 상자 그림
상자는 가운데 절반(1사분위수\~3사분위수), 가로선은 중앙값, 동그라미는 이상치로 볼 만한 값입니다.

In [ ]:
sns.boxplot(data=df, x="species", y="body_mass_g")
plt.show()

### 산점도 행렬 (조금 시간이 걸립니다)

In [ ]:
sns.pairplot(df, hue="species")
plt.show()

## 3. 상관관계
상관계수는 두 특징의 직선 관계를 -1\~1로 나타냅니다.

In [ ]:
num = ["bill_length_mm", "bill_depth_mm",
       "flipper_length_mm", "body_mass_g"]
corr = df[num].corr().round(2)
sns.heatmap(corr, annot=True, cmap="RdBu_r", vmin=-1, vmax=1)
plt.show()

전체로는 부리 깊이와 몸무게가 음의 상관(-0.47)이지만, 종별로 나누면 어떨까요?

In [ ]:
cols = ["bill_depth_mm", "body_mass_g"]
df.groupby("species")[cols].corr().round(2)

세 종 모두 양의 상관입니다. 무리를 합쳤을 때와 나누었을 때 관계가 뒤집히는 현상을 **심슨의 역설**이라고 합니다.

## 4. 빈 값 처리하기

In [ ]:
df.isna().sum()

In [ ]:
cols = ["species", "island", "bill_length_mm", "sex"]
df.loc[df["bill_length_mm"].isna(), cols]

빈 칸을 중앙값으로 채워 봅니다. (여기서는 보여 주기 위해 전체 데이터의 중앙값을 썼지만, 모델을 만들 때는 학습 데이터의 중앙값으로 채웁니다.)

In [ ]:
med = df["bill_length_mm"].median()
filled = df["bill_length_mm"].fillna(med)    # 빈 칸을 중앙값으로
print("채운 값:", med)
print("남은 빈 칸:", filled.isna().sum())

## 5. 범주형 데이터: 원-핫 인코딩
범주에 0, 1, 2 번호를 붙이면 없던 순서와 거리가 생깁니다. 범주마다 열을 하나씩 만듭니다.

In [ ]:
pd.get_dummies(df["island"]).head(3)

## 6. 스케일링: 펭귄 성별 맞히기
성별이 빈 11마리는 빼고, 측정값 네 개로 k-최근접 이웃을 학습합니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

data = df.dropna(subset=["sex"])     # 레이블(성별)이 빈 행은 뺀다
X = data[num]
y = data["sex"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)
print(f"스케일링 전 테스트 정확도: {knn.score(Xte, yte):.1%}")

몸무게의 단위가 커서 거리 계산을 거의 혼자 정합니다. 표준화로 크기를 맞춰 봅시다.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 평균·표준편차는 학습 데이터로만 계산한다
scaler = StandardScaler().fit(Xtr)
Ztr = scaler.transform(Xtr)
Zte = scaler.transform(Xte)    # 테스트 데이터도 같은 값으로 변환

knn = KNeighborsClassifier(n_neighbors=5).fit(Ztr, ytr)
print(f"스케일링 후 테스트 정확도: {knn.score(Zte, yte):.1%}")

## 7. 파이프라인
빈 값 채우기, 스케일링, 원-핫 인코딩, 모델을 하나로 묶습니다. `fit()` 한 번으로 모든 단계가 학습 데이터로만 학습되므로 데이터 누수를 막을 수 있습니다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

cat = ["species", "island"]
X = data[num + cat]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

prep = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"),
                   StandardScaler()), num),            # 숫자 열
    # 글자 열 (처음 보는 범주는 무시)
    (OneHotEncoder(handle_unknown="ignore"), cat))
model = make_pipeline(prep, KNeighborsClassifier(n_neighbors=5))

model.fit(Xtr, ytr)
print(f"테스트 정확도: {model.score(Xte, yte):.1%}")

측정값 하나가 빈 새 펭귄도 예측할 수 있습니다.

In [ ]:
new = pd.DataFrame({
    "bill_length_mm": [45.5], "bill_depth_mm": [14.2],
    "flipper_length_mm": [212.0], "body_mass_g": [np.nan],
    "species": ["Gentoo"], "island": ["Biscoe"]})
print("예측한 성별:", model.predict(new)[0])

---
## 실습 문제

**문제 1.** 부리 깊이(`bill_depth_mm`)의 종별 히스토그램을 그려 보세요. 부리 깊이만으로 잘 골라낼 수 있는 종은 무엇인가요?

In [ ]:
# 문제 1 풀이

**문제 2.** 부리 길이와 부리 깊이의 상관계수를 전체 데이터로 한 번, 종별로 한 번 구해 보세요. 결과를 본문의 부리 깊이–몸무게 예와 비교해 설명하세요.

In [ ]:
# 문제 2 풀이

**문제 3.** 6절(스케일링)에서 `StandardScaler` 대신 `MinMaxScaler`(`from sklearn.preprocessing import MinMaxScaler`)를 써서 성별을 맞혀 보세요. 테스트 정확도는 어떻게 되나요?

In [ ]:
# 문제 3 풀이

**문제 4.** 7절(파이프라인)에서 글자 열(`cat`)을 ① 종만 ② 섬만 ③ 쓰지 않음으로 바꿔 테스트 정확도를 비교해 보세요. 종과 섬은 성별을 맞히는 데 얼마나 도움이 되나요?

In [ ]:
# 문제 4 풀이